In [38]:
import cv2
import numpy as np
import pandas as pd
import glob
from collections import defaultdict
import tqdm
import ast
import copy
import math

In [3]:
experiment_no = 281

In [37]:
def rescale_frame(frame, percent=75):
    width = int(frame.shape[1] * percent/ 100)
    height = int(frame.shape[0] * percent/ 100)
    dim = (width, height)
    return cv2.resize(frame, dim, interpolation =cv2.INTER_AREA)

def find_midpoint(p1, p2):
    """
    Calculates the midpoint between two 2D points.

    Args:
        p1: A tuple or list representing the first point (x1, y1).
        p2: A tuple or list representing the second point (x2, y2).

    Returns:
        A tuple representing the midpoint (xm, ym).
    """
    x1, y1 = p1
    x2, y2 = p2

    xm = (x1 + x2) / 2
    ym = (y1 + y2) / 2

    return (xm, ym)

In [36]:
# loading the MUSE output for the experiment
ssl_data_path = f"D:/big_setup/experiment_{experiment_no}/concatenated_data_cam_mic_sync/ssl_data_path/"
# getting the folder names
folders = glob.glob(ssl_data_path+"*")

data_dict = {}
for folder in folders:
        idx = folder.split("\\")[-1].split("_")[-1]
        try:
            muse_data = pd.read_csv(folder+"/MUSE_pred.txt",header=None)
            muse_data.columns = ["2d_pred","3d_pred","start_time","stop_time","folder"]
            vid = glob.glob(folder+"/*.mp4")[0]
            sleap_track =  pd.read_csv(glob.glob(folder+"/*tracks.csv")[0])
            data_dict[idx] = {"muse":muse_data,"vid":vid,"sleap_track":sleap_track}
        except:
            continue


In [34]:
track_info = defaultdict(dict)
for idx, data in data_dict.items():
    start_cam_frames = (data["muse"]["start_time"]*30).astype('int')
    stop_cam_frames = (data["muse"]["stop_time"]*30).astype('int')
    sleap_tracks = data["sleap_track"]
    # getting the tracks and corresponding frame from video
    for vox_idx,(start,stop) in tqdm.tqdm(enumerate(zip(start_cam_frames,stop_cam_frames))):
        for frame_no in range(start,stop):
            tracks = sleap_tracks[sleap_tracks['frame_idx'] == frame_no]
            try:
                gerbi_0 = [[tracks.iloc[0]["head.x"],tracks.iloc[0]["head.y"]],[tracks.iloc[0]["nose.x"],tracks.iloc[0]["nose.y"]]]
                gerbi_1 = [[tracks.iloc[1]["head.x"],tracks.iloc[1]["head.y"]],[tracks.iloc[1]["nose.x"],tracks.iloc[1]["nose.y"]]]
            except:
                continue

            if not(len(tracks) == 2) or np.any(np.isnan(gerbi_0)) or np.any(np.isnan(gerbi_1)):
                    continue
            else:
                cap = cv2.VideoCapture(data["vid"])

                if not cap.isOpened():
                    print(f"Error: Could not open video file")
                    continue

                # Set the frame position
                cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)

                # Read the frame
                ret, frame = cap.read()

                # Release the VideoCapture object
                cap.release()

                if ret:
                    track_info[idx][vox_idx] = {"frame":frame,"frame_no":frame_no,"track":{"gerbi_0":gerbi_0, "gerbi_1":gerbi_1},"muse_pred":data["muse"]["2d_pred"][vox_idx]}
                    break
                else:
                    continue
                    


19it [00:00, 19.59it/s]
35it [00:01, 20.42it/s]
29it [00:01, 17.59it/s]
37it [00:01, 21.12it/s]
13it [00:00, 25.33it/s]
81it [00:03, 20.98it/s]
52it [00:01, 27.01it/s]
57it [00:02, 20.33it/s]
103it [00:05, 19.59it/s]
14it [00:00, 23.49it/s]
27it [00:01, 21.12it/s]
38it [00:01, 23.71it/s]
22it [00:00, 29.47it/s]
51it [00:02, 21.70it/s]
25it [00:00, 31.81it/s]
36it [00:01, 19.85it/s]
12it [00:00, 17.93it/s]
27it [00:01, 20.17it/s]
14it [00:00, 16.92it/s]
12it [00:00, 20.58it/s]
6it [00:00, 17.04it/s]
50it [00:02, 18.65it/s]
41it [00:01, 24.38it/s]
35it [00:00, 46.51it/s]
12it [00:00, 16.43it/s]
36it [00:01, 20.80it/s]
39it [00:01, 23.02it/s]
36it [00:01, 22.23it/s]
23it [00:00, 30.16it/s]
22it [00:00, 26.67it/s]
46it [00:01, 23.82it/s]
15it [00:00, 19.45it/s]
11it [00:00, 62.82it/s]
5it [00:00, 80.04it/s]


In [41]:
# plot to verify everything
colors = [(255,0,0),(0,255,0),(0,0,255)]
flag = 0
for folder,data in track_info.items():
    for vox_idx,info in data.items():
        image = copy.deepcopy(info["frame"])
        gerb_0 = info["track"]["gerbi_0"]
        gerb_1 = info["track"]["gerbi_1"]
        # Draw the line on the frame
        cv2.arrowedLine(image, np.array(gerb_0[0]).astype(int), np.array(gerb_0[1]).astype(int), colors[1], 1, tipLength=1)
        cv2.arrowedLine(image, np.array(gerb_1[0]).astype(int), np.array(gerb_1[1]).astype(int), colors[1], 1, tipLength=1)

        gerb_0_mid = find_midpoint(np.array(gerb_0[0]), np.array(gerb_0[1]))
        gerb_1_mid = find_midpoint(np.array(gerb_1[0]), np.array(gerb_1[1]))

        muse_pred = list(map(float,info['muse_pred'].strip("[]").split()))

        gerb_0_dist = math.dist(gerb_0_mid, muse_pred)
        gerb_1_dist = math.dist(gerb_1_mid, muse_pred)

        if gerb_0_dist < gerb_1_dist:
            cv2.circle(image, tuple(map(int,gerb_0_mid)), radius=10, color=colors[2], thickness=-1)
        else:
            cv2.circle(image, tuple(map(int,gerb_1_mid)), radius=10, color=colors[2], thickness=-1)

        cv2.circle(image, tuple(map(int,muse_pred)), radius=5, color=colors[0], thickness=-1,)
        while True:
            resized_img = rescale_frame(image,75)
            cv2.imshow("Image",resized_img)
            key = cv2.waitKey(0)
            if key & 0xFF == ord('n'):
                break
            elif key & 0xFF == ord('q'):
                flag = 1
                break
        if flag == 1:
            break
    if flag == 1:
        break

cv2.destroyAllWindows()
        

In [11]:
folder

'000'

In [12]:
gerb_0

[[np.float64(756.9979248046875), np.float64(684.1653442382812)],
 [np.float64(740.7686767578125), np.float64(684.7922973632812)]]

In [ ]:
# int(float(info['muse_pred'].split("[")[-1].split("]")[0].split( )[0])),int(float(info['muse_pred'].split("[")[-1].split("]")[0].split( )[1]))


(1245, 936)